In [1]:
# ==========================================================
# PART 1 : DATA PREPROCESSING
# ==========================================================

import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler

from google.colab import files

# ----------------------------------------------------------
# Upload Dataset
# ----------------------------------------------------------

uploaded = files.upload()

df = pd.read_csv(next(iter(uploaded)))

print("Dataset Shape:", df.shape)

Saving BTCUSDT_Depth_1Hour.csv to BTCUSDT_Depth_1Hour.csv
Dataset Shape: (35901, 45)


In [2]:
# ==========================================================
# PART 1 : DATA PREPROCESSING
# ==========================================================

import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler

# ----------------------------------------------------------
# Load Dataset
# ----------------------------------------------------------

df = pd.read_csv("BTCUSDT_Depth_1Hour.csv")

print("Dataset Shape:", df.shape)

# ----------------------------------------------------------
# Mid Price
# ----------------------------------------------------------

df["mid_price"] = (
    df["best_bid"] + df["best_ask"]
) / 2

# ----------------------------------------------------------
# Log Returns
# ----------------------------------------------------------

df["log_return"] = np.log(
    df["mid_price"] /
    df["mid_price"].shift(1)
)

# ----------------------------------------------------------
# Rolling Volatility
# (20 Snapshot Window)
# ----------------------------------------------------------

window = 20

df["volatility"] = (
    df["log_return"]
    .rolling(window)
    .std()
)

# ----------------------------------------------------------
# Future Volatility
# (Prediction Horizon = 5)
# ----------------------------------------------------------

prediction_horizon = 5

df["future_volatility"] = (
    df["volatility"]
    .shift(-prediction_horizon)
)

# ----------------------------------------------------------
# Create Labels
# ----------------------------------------------------------

low = df["future_volatility"].quantile(0.33)

high = df["future_volatility"].quantile(0.66)

df["label"] = np.where(

    df["future_volatility"] < low,

    0,

    np.where(

        df["future_volatility"] < high,

        1,

        2
    )
)

# ----------------------------------------------------------
# Remove Missing Values
# ----------------------------------------------------------

df = df.dropna().reset_index(drop=True)

# ----------------------------------------------------------
# Level-10 LOB Features
# ----------------------------------------------------------

lob_features = [

    col for col in df.columns

    if ("bid" in col.lower()) or
       ("ask" in col.lower())

]

X = df[lob_features]

y = df["label"]

# ----------------------------------------------------------
# Feature Scaling
# ----------------------------------------------------------

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

# ----------------------------------------------------------
# Check Results
# ----------------------------------------------------------

print("\nFeature Matrix Shape:", X_scaled.shape)

print("\nLabel Distribution")

print(y.value_counts().sort_index())

Dataset Shape: (35901, 45)

Feature Matrix Shape: (4652, 44)

Label Distribution
label
0    2277
1    1367
2    1008
Name: count, dtype: int64


In [3]:
# ==========================================================
# PART 2 : SEQUENCE GENERATION
# ==========================================================

from sklearn.model_selection import train_test_split

# ----------------------------------------------------------
# Sequence Length
# ----------------------------------------------------------

sequence_length = 50

X_seq = []
y_seq = []

for i in range(sequence_length, len(X_scaled)):

    X_seq.append(
        X_scaled[i-sequence_length:i]
    )

    y_seq.append(
        y.iloc[i]
    )

X_seq = np.array(X_seq)
y_seq = np.array(y_seq)

print("Sequence Shape:", X_seq.shape)

# ----------------------------------------------------------
# Train-Test Split
# ----------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(

    X_seq,
    y_seq,

    test_size=0.20,

    random_state=42,

    stratify=y_seq

)

print("\nTraining Shape :", X_train.shape)
print("Testing Shape  :", X_test.shape)

print("\nTraining Label Distribution")
print(pd.Series(y_train).value_counts().sort_index())

print("\nTesting Label Distribution")
print(pd.Series(y_test).value_counts().sort_index())

Sequence Shape: (4602, 50, 44)

Training Shape : (3681, 50, 44)
Testing Shape  : (921, 50, 44)

Training Label Distribution
0    1781
1    1094
2     806
Name: count, dtype: int64

Testing Label Distribution
0    446
1    273
2    202
Name: count, dtype: int64


In [6]:
# ==========================================================
# PART 3 : TRANSFORMER MODEL
# ==========================================================

import tensorflow as tf

from tensorflow.keras.layers import (
    Input,
    MultiHeadAttention,
    LayerNormalization,
    Dense,
    Dropout,
    GlobalAveragePooling1D
)

from tensorflow.keras.models import Model

# ----------------------------------------------------------
# Input
# ----------------------------------------------------------

inputs = Input(shape=(X_train.shape[1], X_train.shape[2]))

# ----------------------------------------------------------
# Multi-Head Attention
# ----------------------------------------------------------

attention = MultiHeadAttention(
    num_heads=4,
    key_dim=16
)(inputs, inputs)

x = LayerNormalization()(inputs + attention)

# ----------------------------------------------------------
# Feed Forward Network
# ----------------------------------------------------------

ffn = Dense(64, activation="relu")(x)

ffn = Dense(X_train.shape[2])(ffn)

x = LayerNormalization()(x + ffn)

# ----------------------------------------------------------
# Pooling
# ----------------------------------------------------------

x = GlobalAveragePooling1D()(x)

# ----------------------------------------------------------
# Classification Head
# ----------------------------------------------------------

x = Dropout(0.30)(x)

x = Dense(32, activation="relu")(x)

outputs = Dense(3, activation="softmax")(x)

# ----------------------------------------------------------
# Build Model
# ----------------------------------------------------------

model = Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 50, 44)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 50, 44)    │     11,500 │ input_layer_2[0]… │
│ (MultiHeadAttentio… │                   │            │ input_layer_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_4 (Add)         │ (None, 50, 44)    │          0 │ input_layer_2[0]… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 50, 44)    │         88 │ add_4[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_8 (Dense)     │ (None, 50, 64)    │      2,880 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_9 (Dense)     │ (None, 50, 44)    │      2,860 │ dense_8[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_5 (Add)         │ (None, 50, 44)    │          0 │ layer_normalizat… │
│                     │                   │            │ dense_9[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 50, 44)    │         88 │ add_5[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 44)        │          0 │ layer_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 44)        │          0 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_10 (Dense)    │ (None, 32)        │      1,440 │ dropout_5[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_11 (Dense)    │ (None, 3)         │         99 │ dense_10[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 18,955 (74.04 KB)

 Trainable params: 18,955 (74.04 KB)

 Non-trainable params: 0 (0.00 B)

In [7]:
# ==========================================================
# PART 4 : OPTUNA
# ==========================================================

!pip install -q optuna

import optuna
import tensorflow as tf

from tensorflow.keras.layers import (
    Input,
    MultiHeadAttention,
    LayerNormalization,
    Dense,
    Dropout,
    GlobalAveragePooling1D
)

from tensorflow.keras.models import Model

# ----------------------------------------------------------
# Objective Function
# ----------------------------------------------------------

def objective(trial):

    num_heads = trial.suggest_categorical(
        "num_heads",
        [2,4,8]
    )

    key_dim = trial.suggest_categorical(
        "key_dim",
        [8,16,32]
    )

    ff_units = trial.suggest_categorical(
        "ff_units",
        [32,64,128]
    )

    dropout = trial.suggest_float(
        "dropout",
        0.2,
        0.5
    )

    learning_rate = trial.suggest_float(
        "learning_rate",
        1e-4,
        1e-2,
        log=True
    )

    batch_size = trial.suggest_categorical(
        "batch_size",
        [32,64,128]
    )

    # -----------------------------
    # Model
    # -----------------------------

    inputs = Input(shape=(X_train.shape[1], X_train.shape[2]))

    attention = MultiHeadAttention(
        num_heads=num_heads,
        key_dim=key_dim
    )(inputs, inputs)

    x = LayerNormalization()(inputs + attention)

    ffn = Dense(
        ff_units,
        activation="relu"
    )(x)

    ffn = Dense(X_train.shape[2])(ffn)

    x = LayerNormalization()(x + ffn)

    x = GlobalAveragePooling1D()(x)

    x = Dropout(dropout)(x)

    x = Dense(
        32,
        activation="relu"
    )(x)

    outputs = Dense(
        3,
        activation="softmax"
    )(x)

    model = Model(inputs, outputs)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=learning_rate
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    history = model.fit(
        X_train,
        y_train,
        validation_split=0.2,
        epochs=10,
        batch_size=batch_size,
        verbose=0
    )

    return max(history.history["val_accuracy"])

# ----------------------------------------------------------
# Run Optuna
# ----------------------------------------------------------

study = optuna.create_study(direction="maximize")

study.optimize(objective, n_trials=10)

print("\nBest Parameters")
print(study.best_params)

print("\nBest Validation Accuracy")
print(study.best_value)

[I 2026-07-28 12:39:04,475] A new study created in memory with name: no-name-8d926eca-0a02-468e-a5bc-d7c6ea075f19
[I 2026-07-28 12:39:18,386] Trial 0 finished with value: 0.5725916028022766 and parameters: {'num_heads': 4, 'key_dim': 8, 'ff_units': 32, 'dropout': 0.2377758002684588, 'learning_rate': 0.00034039110057119216, 'batch_size': 32}. Best is trial 0 with value: 0.5725916028022766.
[I 2026-07-28 12:39:33,675] Trial 1 finished with value: 0.5793758630752563 and parameters: {'num_heads': 4, 'key_dim': 32, 'ff_units': 128, 'dropout': 0.38227506915571874, 'learning_rate': 0.003054247640196648, 'batch_size': 128}. Best is trial 1 with value: 0.5793758630752563.
[I 2026-07-28 12:39:44,658] Trial 2 finished with value: 0.5671641826629639 and parameters: {'num_heads': 8, 'key_dim': 32, 'ff_units': 128, 'dropout': 0.36725113966001477, 'learning_rate': 0.0007963568135409527, 'batch_size': 128}. Best is trial 1 with value: 0.5793758630752563.
[I 2026-07-28 12:40:00,802] Trial 3 finished wi


Best Parameters
{'num_heads': 8, 'key_dim': 8, 'ff_units': 128, 'dropout': 0.2110346469249726, 'learning_rate': 0.008312961912019213, 'batch_size': 64}

Best Validation Accuracy
0.6295793652534485


In [8]:
# ==========================================================
# PART 5 : FINAL TRANSFORMER TRAINING
# ==========================================================

import tensorflow as tf

from tensorflow.keras.layers import (
    Input,
    MultiHeadAttention,
    LayerNormalization,
    Dense,
    Dropout,
    GlobalAveragePooling1D
)

from tensorflow.keras.models import Model

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)

# ----------------------------------------------------------
# Best Parameters from Optuna
# ----------------------------------------------------------

num_heads = study.best_params["num_heads"]

key_dim = study.best_params["key_dim"]

ff_units = study.best_params["ff_units"]

dropout = study.best_params["dropout"]

learning_rate = study.best_params["learning_rate"]

batch_size = study.best_params["batch_size"]

# ----------------------------------------------------------
# Build Final Model
# ----------------------------------------------------------

inputs = Input(shape=(X_train.shape[1], X_train.shape[2]))

attention = MultiHeadAttention(
    num_heads=num_heads,
    key_dim=key_dim
)(inputs, inputs)

x = LayerNormalization()(inputs + attention)

ffn = Dense(ff_units, activation="relu")(x)

ffn = Dense(X_train.shape[2])(ffn)

x = LayerNormalization()(x + ffn)

x = GlobalAveragePooling1D()(x)

x = Dropout(dropout)(x)

x = Dense(32, activation="relu")(x)

outputs = Dense(3, activation="softmax")(x)

model = Model(inputs, outputs)

model.compile(

    optimizer=tf.keras.optimizers.Adam(
        learning_rate=learning_rate
    ),

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]

)

# ----------------------------------------------------------
# Callbacks
# ----------------------------------------------------------

early_stop = EarlyStopping(

    monitor="val_loss",

    patience=5,

    restore_best_weights=True

)

reduce_lr = ReduceLROnPlateau(

    monitor="val_loss",

    factor=0.5,

    patience=3,

    verbose=1

)

checkpoint = ModelCheckpoint(

    "best_transformer.keras",

    monitor="val_accuracy",

    save_best_only=True,

    verbose=1

)

# ----------------------------------------------------------
# Train Model
# ----------------------------------------------------------

history = model.fit(

    X_train,

    y_train,

    validation_split=0.2,

    epochs=30,

    batch_size=batch_size,

    callbacks=[
        early_stop,
        reduce_lr,
        checkpoint
    ],

    verbose=1

)

Epoch 1/30
41/46 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.4537 - loss: 1.0966
Epoch 1: val_accuracy improved from None to 0.48847, saving model to best_transformer.keras

Epoch 1: finished saving model to best_transformer.keras
46/46 ━━━━━━━━━━━━━━━━━━━━ 8s 41ms/step - accuracy: 0.4701 - loss: 1.0433 - val_accuracy: 0.4885 - val_loss: 1.0291 - learning_rate: 0.0083
Epoch 2/30
43/46 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5256 - loss: 0.9514
Epoch 2: val_accuracy improved from 0.48847 to 0.51153, saving model to best_transformer.keras

Epoch 2: finished saving model to best_transformer.keras
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5132 - loss: 0.9587 - val_accuracy: 0.5115 - val_loss: 0.9580 - learning_rate: 0.0083
Epoch 3/30
45/46 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5010 - loss: 0.9415
Epoch 3: val_accuracy improved from 0.51153 to 0.52782, saving model to best_transformer.keras

Epoch 3: finished saving model to best_transformer.keras
46/46 ━━━━━━━

In [9]:
# ==========================================================
# PART 6 : MODEL EVALUATION
# ==========================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import numpy as np

# ----------------------------------------------------------
# Predictions
# ----------------------------------------------------------

y_pred_prob = model.predict(X_test)

y_pred = np.argmax(y_pred_prob, axis=1)

# ----------------------------------------------------------
# Metrics
# ----------------------------------------------------------

accuracy = accuracy_score(y_test, y_pred)

precision = precision_score(
    y_test,
    y_pred,
    average="weighted"
)

recall = recall_score(
    y_test,
    y_pred,
    average="weighted"
)

f1 = f1_score(
    y_test,
    y_pred,
    average="weighted"
)

print("\nAccuracy :", round(accuracy*100,2), "%")
print("Precision:", round(precision*100,2), "%")
print("Recall   :", round(recall*100,2), "%")
print("F1 Score :", round(f1*100,2), "%")

# ----------------------------------------------------------
# Classification Report
# ----------------------------------------------------------

print("\nClassification Report\n")

print(classification_report(y_test, y_pred))

# ----------------------------------------------------------
# Confusion Matrix
# ----------------------------------------------------------

print("\nConfusion Matrix\n")

print(confusion_matrix(y_test, y_pred))

29/29 ━━━━━━━━━━━━━━━━━━━━ 4s 36ms/step

Accuracy : 64.93 %
Precision: 66.69 %
Recall   : 64.93 %
F1 Score : 65.46 %

Classification Report

              precision    recall  f1-score   support

           0       0.80      0.68      0.73       446
           1       0.55      0.63      0.59       273
           2       0.53      0.61      0.57       202

    accuracy                           0.65       921
   macro avg       0.63      0.64      0.63       921
weighted avg       0.67      0.65      0.65       921


Confusion Matrix

[[303  90  53]
 [ 47 172  54]
 [ 31  48 123]]
